In [13]:
import os
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Trening na urządzeniu: {device}")
if torch.cuda.is_available():
    print(f"Karta: {torch.cuda.get_device_name(0)}")

Trening na urządzeniu: cuda
Karta: NVIDIA GeForce RTX 5060 Laptop GPU


In [15]:
import tonic
import torch.nn as nn
import random
import pandas as pd
import numpy as np
import math

In [16]:
transform_dataset = tonic.transforms.Compose([
    tonic.transforms.Downsample(spatial_factor=0.5)
])

transform_train = tonic.transforms.Compose([
    tonic.transforms.RandomCrop(sensor_size=(120,90,2), target_size=(80,80)),
    tonic.transforms.RandomFlipLR(sensor_size=(80,80,2), p = 0.5),
    tonic.transforms.EventDrop(sensor_size=(80,80,2)),
    tonic.transforms.ToFrame(sensor_size=(80,80,2), n_time_bins=30)
])

transform_val = tonic.transforms.Compose([
    tonic.transforms.CenterCrop(sensor_size=(120,90,2), size=(80,80)),
    tonic.transforms.ToFrame(sensor_size=(80,80,2), n_time_bins=30)
])

In [17]:
dataset = tonic.datasets.NCALTECH101(save_to = './Data_80_80' , transform=transform_dataset)

In [18]:
#changing the targets to numerical values:
classes = sorted(set(dataset.targets))
class_to_idx = {cls: i for i, cls in enumerate(classes)}
dataset.targets = [class_to_idx[target] for target in dataset.targets]

In [19]:
from torch.utils.data import random_split

train = int(len(dataset) * 0.7)
val = int(len(dataset) * 0.2)
test = len(dataset) - train - val

train, val, test = random_split(dataset, [train, val, test])
len(train), len(val), len(test)

(6096, 1741, 872)

In [20]:
batch_size = 32

In [21]:
from torch.utils.data import DataLoader, Dataset
from tonic.cached_dataset import MemoryCachedDataset

class SafeDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        
    def __getitem__(self, idx):
        data, target = self.dataset[idx]
        while isinstance(data, np.ndarray) and data.dtype.names is not None:
            idx = np.random.randint(0, len(self.dataset))
            data, target = self.dataset[idx]
        return data, target

    def __len__(self):
        return len(self.dataset)

train_cached = tonic.MemoryCachedDataset(train, transform=transform_train)
val_cached = tonic.MemoryCachedDataset(val, transform=transform_val)
test_cached = tonic.MemoryCachedDataset(test, transform=transform_val)

train_set_safe = SafeDataset(train_cached)
val_set_safe = SafeDataset(val_cached)
test_set_safe = SafeDataset(test_cached)

train_loader = DataLoader(train_set_safe, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False),
                            shuffle=True, drop_last=True, pin_memory=True)
val_loader = DataLoader(val_set_safe, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False),
                            drop_last=True, pin_memory=True)
test_loader = DataLoader(test_set_safe, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False),
                            drop_last=True, pin_memory=True)

In [22]:
event, target = train_cached[200] 
print(f'Output: {event.shape} for target {target}')

Output: (30, 2, 80, 80) for target 1


In [24]:
import datetime
import os
import time
import torch
import torch.utils.data
from torch import nn
import torchvision
from torchvision import transforms
import math
from torch.cuda import amp
import torch.distributed.optim
import argparse

from spikingjelly.clock_driven import functional
import spiking_resnet, sew_resnet, utils

In [25]:
def train_one_epoch(model, criterion, optimizer, data_loader, device, epoch, print_freq, scaler=None):
    model.train()
    metric_logger = utils.MetricLogger(delimiter="  ")
    metric_logger.add_meter('lr', utils.SmoothedValue(window_size=1, fmt='{value}'))
    metric_logger.add_meter('img/s', utils.SmoothedValue(window_size=10, fmt='{value}'))

    header = 'Epoch: [{}]'.format(epoch)

    for image, target in metric_logger.log_every(data_loader, print_freq, header):
        start_time = time.time()
        image, target = image.to(device), target.to(device)
        # with torch.autograd.detect_anomaly():
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                output = model(image)
                loss = criterion(output, target)
        else:
            output = model(image)
            loss = criterion(output, target)

        optimizer.zero_grad()

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:
            loss.backward()
            optimizer.step()

        functional.reset_net(model)

        acc1, acc5 = utils.accuracy(output, target, topk=(1, 5))
        batch_size = image.shape[0]
        loss_s = loss.item()
        if math.isnan(loss_s):
            raise ValueError('loss is Nan')
        acc1_s = acc1.item()
        acc5_s = acc5.item()

        metric_logger.update(loss=loss_s, lr=optimizer.param_groups[0]["lr"])

        metric_logger.meters['acc1'].update(acc1_s, n=batch_size)
        metric_logger.meters['acc5'].update(acc5_s, n=batch_size)
        metric_logger.meters['img/s'].update(batch_size / (time.time() - start_time))

    # gather the stats from all processes
    metric_logger.synchronize_between_processes()
    return metric_logger.loss.global_avg, metric_logger.acc1.global_avg, metric_logger.acc5.global_avg



def evaluate(model, criterion, data_loader, device, print_freq=100, header='Test:'):
    model.eval()
    metric_logger = utils.MetricLogger(delimiter="  ")
    with torch.no_grad():
        for image, target in metric_logger.log_every(data_loader, print_freq, header):
            image = image.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            output = model(image)
            loss = criterion(output, target)
            functional.reset_net(model)

            acc1, acc5 = utils.accuracy(output, target, topk=(1, 5))
            # FIXME need to take into account that the datasets
            # could have been padded in distributed setup
            batch_size = image.shape[0]
            metric_logger.update(loss=loss.item())
            metric_logger.meters['acc1'].update(acc1.item(), n=batch_size)
            metric_logger.meters['acc5'].update(acc5.item(), n=batch_size)
    # gather the stats from all processes
    metric_logger.synchronize_between_processes()

    loss, acc1, acc5 = metric_logger.loss.global_avg, metric_logger.acc1.global_avg, metric_logger.acc5.global_avg
    print(f' * Acc@1 = {acc1}, Acc@5 = {acc5}, loss = {loss}')
    return loss, acc1, acc5


In [26]:
def main(args):


    max_test_acc1 = 0.
    test_acc5_at_max_test_acc1 = 0.


    utils.init_distributed_mode(args)
    print(args)
    output_dir = os.path.join(args.output_dir, f'{args.model}_b{args.batch_size}_lr{args.lr}_T{args.T}')

    if args.zero_init_residual:
        output_dir += '_zi'
    if args.weight_decay:
        output_dir += f'_wd{args.weight_decay}'

    output_dir += f'_coslr{args.cos_lr_T}'

    if args.adam:
        output_dir += '_adam'
    else:
        output_dir += '_sgd'

    if args.connect_f:
        output_dir += f'_cnf_{args.connect_f}'

    if output_dir:
        utils.mkdir(output_dir)
        if args.tb and utils.is_main_process():
            utils.mkdir(os.path.join(output_dir, '_logs'))


    device = torch.device(args.device)


    print("Creating model")

    if args.model in sew_resnet.__dict__:
        model = sew_resnet.__dict__[args.model](zero_init_residual=args.zero_init_residual, T=args.T, connect_f=args.connect_f)
    elif args.model in spiking_resnet.__dict__:
        model = spiking_resnet.__dict__[args.model](zero_init_residual=args.zero_init_residual, T=args.T)
    else:
        raise NotImplementedError(args.model)

    print(model)

    model.to(device)
    if args.distributed and args.sync_bn:
        model = torch.nn.SyncBatchNorm.convert_sync_batchnorm(model)

    criterion = nn.CrossEntropyLoss()


    if args.adam:
        optimizer = torch.optim.Adam(
            model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    else:
        optimizer = torch.optim.SGD(
            model.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.weight_decay)

    if args.amp:
        scaler = torch.amp.GradScaler('cuda')
    else:
        scaler = None

    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.cos_lr_T)



    model_without_ddp = model
    if args.distributed:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu])
        model_without_ddp = model.module

    if args.resume:

        checkpoint = torch.load(args.resume, map_location='cpu', weights_only=False)
        model_without_ddp.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])

        args.start_epoch = checkpoint['epoch'] + 1

        max_test_acc1 = checkpoint['max_test_acc1']
        test_acc5_at_max_test_acc1 = checkpoint['test_acc5_at_max_test_acc1']

    if args.test_only:
        evaluate(model, criterion, val_loader, device=device, header='Test:')
        return


    print("Start training")
    start_time = time.time()
    for epoch in range(args.start_epoch, args.epochs):
        save_max = False
        if args.distributed:
            train_loader.set_epoch(epoch)
        train_loss, train_acc1, train_acc5 = train_one_epoch(model, criterion, optimizer, train_loader, device, epoch, args.print_freq, scaler)
       
        lr_scheduler.step()

        test_loss, test_acc1, test_acc5 = evaluate(model, criterion, val_loader, device=device, header='Test:')

        if max_test_acc1 < test_acc1:
            max_test_acc1 = test_acc1
            test_acc5_at_max_test_acc1 = test_acc5
            save_max = True



        if output_dir:

            checkpoint = {
                'model': model_without_ddp.state_dict(),
                'optimizer': optimizer.state_dict(),
                'lr_scheduler': lr_scheduler.state_dict(),
                'epoch': epoch,
                'args': args,
                'max_test_acc1': max_test_acc1,
                'test_acc5_at_max_test_acc1': test_acc5_at_max_test_acc1,
            }

            utils.save_on_master(
                checkpoint,
                os.path.join(output_dir, 'checkpoint_latest.pth'))
            save_flag = False

            if epoch % 64 == 0 or epoch == args.epochs - 1:
                save_flag = True

            elif args.cos_lr_T == 0:
                for item in args.lr_step_size:
                    if (epoch + 2) % item == 0:
                        save_flag = True
                        break

            if save_flag:
                utils.save_on_master(
                    checkpoint,
                    os.path.join(output_dir, f'checkpoint_{epoch}.pth'))

            if save_max:
                utils.save_on_master(
                    checkpoint,
                    os.path.join(output_dir, 'checkpoint_max_test_acc1.pth'))
        print(args)
        total_time = time.time() - start_time
        total_time_str = str(datetime.timedelta(seconds=int(total_time)))
        print(output_dir)

        print('Training time {}'.format(total_time_str), 'max_test_acc1', max_test_acc1,
              'test_acc5_at_max_test_acc1', test_acc5_at_max_test_acc1)


In [ ]:
class Args:

    model = 'sew_resnet18'        
    T = 4                      
    zero_init_residual = False    
    connect_f = 'ADD'             

    batch_size = 32               
    epochs = 100                   
    lr = 0.1                     
    momentum = 0.9                
    weight_decay = 1e-4           
    adam = False                  
    cos_lr_T = 100                 
    lr_step_size = [30, 60]       
    amp = True                    

    device = 'cuda'               
    distributed = False           
    gpu = 0                       
    sync_bn = False               

    resume = 'C:/Users/Veronika/Documents/NCALTECH_VS/resnet/checkpoints/sew_resnet18_b32_lr0.1_T4_wd0.0001_coslr64_sgd_cnf_ADD/checkpoint_63.pth'                  
    test_only = False             
    tb = False                   
    print_freq = 50               
    output_dir = './checkpoints'  
    start_epoch = 0               

args = Args()

In [28]:
main(args)

Not using distributed mode
Creating model
SEWResNet(
  (conv1): SeqToANNContainer(
    (0): Conv2d(2, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  )
  (bn1): SeqToANNContainer(
    (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (sn1): IFNode(
    v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch
    (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
  )
  (maxpool): SeqToANNContainer(
    (0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): SeqToANNContainer(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (sn1): IFNode(
        v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch
        (surrogate_function): Sigmoid(alpha=4.0, s

c:\Users\Veronika\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1370: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


Start training
Epoch: [64]  [  0/190]  eta: 0:07:55  lr: 0.0  img/s: 16.553631615644694  loss: 1.1814 (1.1814)  acc1: 71.8750 (71.8750)  acc5: 84.3750 (84.3750)  time: 2.5052  data: 0.6929  max mem: 4023
Epoch: [64]  [ 50/190]  eta: 0:02:29  lr: 0.0  img/s: 76.82794766650954  loss: 1.0369 (1.1774)  acc1: 71.8750 (69.7917)  acc5: 90.6250 (88.4804)  time: 1.0030  data: 0.5889  max mem: 4055
Epoch: [64]  [100/190]  eta: 0:01:49  lr: 0.0  img/s: 48.73993809368685  loss: 1.1793 (1.1845)  acc1: 71.8750 (69.7401)  acc5: 87.5000 (88.2735)  time: 1.5066  data: 0.7027  max mem: 4055
Epoch: [64]  [150/190]  eta: 0:00:55  lr: 0.0  img/s: 40.05380865585952  loss: 1.1690 (1.1845)  acc1: 68.7500 (69.6192)  acc5: 87.5000 (88.4106)  time: 1.6605  data: 0.8119  max mem: 4055
Epoch: [64] Total time: 0:04:38
Test:  [ 0/54]  eta: 0:05:25  loss: 2.0142 (2.0142)  acc1: 53.1250 (53.1250)  acc5: 78.1250 (78.1250)  time: 6.0299  data: 1.0768  max mem: 4055
Test: Total time: 0:01:43
 * Acc@1 = 57.46527777777778,

KeyboardInterrupt: 

In [41]:
model = sew_resnet.__dict__[args.model](zero_init_residual=args.zero_init_residual, T=args.T, connect_f=args.connect_f)

In [44]:
checkpoint_path = 'C:/Users/Veronika/Documents/NCALTECH_VS/resnet/checkpoints/sew_resnet18_b32_lr0.1_T4_wd0.0001_coslr100_sgd_cnf_ADD/checkpoint_max_test_acc1.pth'
checkpoint = torch.load(checkpoint_path, map_location='cuda', weights_only=False)

In [45]:
model.load_state_dict(checkpoint['model'])
model.eval()
model.to('cuda')

SEWResNet(
  (conv1): SeqToANNContainer(
    (0): Conv2d(2, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  )
  (bn1): SeqToANNContainer(
    (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (sn1): IFNode(
    v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch
    (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
  )
  (maxpool): SeqToANNContainer(
    (0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): SeqToANNContainer(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (sn1): IFNode(
        v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch
        (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
      )
      (conv2): SeqToA

In [47]:
evaluate(model, criterion= nn.CrossEntropyLoss(), data_loader=test_loader, device=torch.device(args.device))

Test:  [ 0/27]  eta: 0:00:23  loss: 2.2340 (2.2340)  acc1: 50.0000 (50.0000)  acc5: 65.6250 (65.6250)  time: 0.8868  data: 0.5725  max mem: 4396
Test: Total time: 0:00:19
 * Acc@1 = 59.49074074074074, Acc@5 = 78.93518518518519, loss = 1.7150370522781655


(1.7150370522781655, 59.49074074074074, 78.93518518518519)